In [3]:
from opensearchpy import OpenSearch
import pandas as pd

client = OpenSearch(hosts=["http://localhost:9201"], use_ssl=False)

# Basic stats
print("Total documents:", client.count(index="riskpulse-logs")["count"])

query = {
    "size": 0,
    "aggs": {
        "by_eventid": {
            "terms": {"field": "EventID", "size": 20}
        },
        "by_hostname": {
            "terms": {"field": "Hostname.keyword", "size": 10}
        },
        "by_technique": {  # If you added source/filename as field
            "terms": {"field": "your_source_field.keyword", "size": 10}
        }
    }
}
response = client.search(index="riskpulse-logs", body=query)
print(response["aggregations"])
for hit in response["hits"]["hits"]:
    print(hit["_source"].get("@timestamp"), "–", hit["_source"].get("EventID"), 
          hit["_source"].get("Channel", ""), hit["_source"].get("Hostname"))

Total documents: 21050
{'by_hostname': {'doc_count_error_upper_bound': 0, 'sum_other_doc_count': 0, 'buckets': [{'key': 'WORKSTATION5.theshire.local', 'doc_count': 9090}, {'key': 'WORKSTATION6.mordor.local', 'doc_count': 4352}, {'key': 'MORDORDC.theshire.local', 'doc_count': 3825}, {'key': 'WORKSTATION6.theshire.local', 'doc_count': 1778}, {'key': 'MORDORDC.mordor.local', 'doc_count': 1169}, {'key': 'WORKSTATION5.mordor.local', 'doc_count': 681}, {'key': 'WORKSTATION6', 'doc_count': 155}]}, 'by_eventid': {'doc_count_error_upper_bound': 3, 'sum_other_doc_count': 374, 'buckets': [{'key': 800, 'doc_count': 4427}, {'key': 4103, 'doc_count': 4098}, {'key': 10, 'doc_count': 3020}, {'key': 12, 'doc_count': 2552}, {'key': 4658, 'doc_count': 1394}, {'key': 13, 'doc_count': 1012}, {'key': 7, 'doc_count': 923}, {'key': 4656, 'doc_count': 698}, {'key': 4690, 'doc_count': 690}, {'key': 4663, 'doc_count': 644}, {'key': 5156, 'doc_count': 254}, {'key': 4703, 'doc_count': 252}, {'key': 5158, 'doc_coun

In [ ]:
# Example labeling logic (you can run this after exporting or in memory)

# Rough initial labeling strategy
def simple_label(event):
    source = str(event)  # or better: use file name/path you ingested from
    
    if "dcsync" in source.lower() or "DsGetNCChanges" in str(event):
        return "malicious"      # credential access
    elif "powerview" in source.lower() or "get_all" in source.lower():
        return "suspicious"     # discovery/recon
    elif "empire" in source.lower() and "invoke" in source.lower():
        return "suspicious"     # execution
    elif any(x in str(event).lower() for x in ["svchost", "lsass", "powershell", "wmic", "psexec"]):
        return "suspicious"
    else:
        return "normal"         # default + your own normal traffic

# Later you'll improve this massively